# Phase 2: Preprocessing

**Goal:** Fit the scaler and encoders on training data, transform both train and test, save all preprocessing objects.

**Input:**
- `data/processed/train_cleaned.csv`
- `data/processed/test_cleaned.csv`

**Output:**
- `data/processed/train_processed.parquet`
- `data/processed/test_processed.parquet`
- `models/scaler.pkl`
- `models/encoder_objects.pkl`
- `data/mappings/<col>.json` for each categorical column

**Rule:** `fit_transform()` is called only on training data. Test data uses `transform()` only. This prevents data leakage.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import os

from src.preprocessor import CreditPreprocessor
from src.data_loader import NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET_COL

## 1. Load Cleaned Data

In [2]:
df_train = pd.read_csv('../data/processed/train_cleaned.csv')
df_test  = pd.read_csv('../data/processed/test_cleaned.csv')

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
df_train.head()

Train shape: (307511, 17)
Test shape:  (48744, 16)


,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_ID_PUBLISH,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,OCCUPATION_TYPE,ORGANIZATION_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_INCOME_TYPE,CODE_GENDER
0,1,202500.0,406597.5,24700.5,351000.0,-9461,-637,-2120,0.083037,0.262949,0.139376,Laborers,Business Entity Type 3,Secondary / secondary special,Single / not married,Working,M
1,0,270000.0,1293502.5,35698.5,1129500.0,-16765,-1188,-291,0.311267,0.622246,NaN,Core staff,School,Higher education,Married,State servant,F
2,0,67500.0,135000.0,6750.0,135000.0,-19046,-225,-2531,NaN,0.555912,0.729567,Laborers,Government,Secondary / secondary special,Single / not married,Working,M
3,0,135000.0,312682.5,29686.5,297000.0,-19005,-3039,-2437,NaN,0.650442,NaN,Laborers,Business Entity Type 3,Secondary / secondary special,Civil marriage,Working,F
4,0,121500.0,513000.0,21865.5,513000.0,-19932,-3038,-3458,NaN,0.322738,NaN,Core staff,Religion,Secondary / secondary special,Single / not married,Working,M


## 2. Fit and Transform Training Data

This is the only place `fit_transform()` is called. The scaler and encoders learn their parameters from training data here and nowhere else.

In [3]:
preprocessor = CreditPreprocessor()
df_train_processed = preprocessor.fit_transform(df_train)

print(f"\nProcessed train shape: {df_train_processed.shape}")
print(f"Columns: {df_train_processed.columns.tolist()}")
df_train_processed.head()

[preprocessor] OCCUPATION_TYPE: 19 unique categories
[preprocessor] ORGANIZATION_TYPE: 58 unique categories
[preprocessor] NAME_EDUCATION_TYPE: 5 unique categories
[preprocessor] NAME_FAMILY_STATUS: 6 unique categories
[preprocessor] NAME_INCOME_TYPE: 8 unique categories
[preprocessor] CODE_GENDER: 3 unique categories

Processed train shape: (307511, 21)
Columns: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'LOAN_TO_GOODS_RATIO', 'EXT_SOURCE_1_MISSING', 'EXT_SOURCE_2_MISSING', 'EXT_SOURCE_3_MISSING', 'AGE_YEARS', 'YEARS_EMPLOYED', 'YEARS_ID_PUBLISH', 'IS_UNEMPLOYED', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE', 'CODE_GENDER', 'TARGET']


,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,LOAN_TO_GOODS_RATIO,EXT_SOURCE_1_MISSING,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING,...,YEARS_EMPLOYED,YEARS_ID_PUBLISH,IS_UNEMPLOYED,OCCUPATION_TYPE,ORGANIZATION_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_INCOME_TYPE,CODE_GENDER,TARGET
0,0.142129,-0.478095,-0.166143,-3.021877,-1.317940,-2.153651,0.286298,0,0,0,...,-0.755835,-0.579154,0,8,5,4,3,7,1,1
1,0.426792,1.725450,0.592683,-1.384737,0.564482,0.112063,0.179898,0,0,1,...,-0.497899,-1.790855,0,3,39,1,1,4,0,0
2,-0.427196,-1.152888,-1.404669,0.012103,0.216948,1.223975,-0.990650,1,0,0,...,-0.948701,-0.306869,0,8,11,4,3,7,1,0
3,-0.142533,-0.711430,0.177874,0.012103,0.712205,0.112063,-0.564969,1,0,1,...,0.368597,-0.369143,0,8,5,4,0,7,0,0
4,-0.199466,-0.213734,-0.361749,0.012103,-1.004691,0.112063,-0.990650,1,0,1,...,0.368129,0.307263,0,3,37,4,3,7,1,0


## 3. Verify Cardinalities

These numbers go into `model_builder.py` as the `cat_feature_cardinalities` argument. They are saved inside `encoder_objects.pkl` so the API can retrieve them automatically.

In [4]:
print("Categorical feature cardinalities:")
for col, n in preprocessor.cardinalities.items():
    print(f"  {col}: {n} unique values")

Categorical feature cardinalities:
  OCCUPATION_TYPE: 19 unique values
  ORGANIZATION_TYPE: 58 unique values
  NAME_EDUCATION_TYPE: 5 unique values
  NAME_FAMILY_STATUS: 6 unique values
  NAME_INCOME_TYPE: 8 unique values
  CODE_GENDER: 3 unique values


## 4. Sanity Checks

In [5]:
# Check 1: No NaNs anywhere
nan_count = df_train_processed.isnull().sum().sum()
print(f"NaN values remaining: {nan_count}")
assert nan_count == 0, "NaNs found after preprocessing!"

# Check 2: Numerical columns are scaled (mean ~0, std ~1)
print("\nScaled column stats (should be ~mean=0, std=1):")
scale_check_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AGE_YEARS']
print(df_train_processed[scale_check_cols].describe().loc[['mean', 'std']].round(3))

# Check 3: Categorical columns are integers
print("\nCategorical dtypes (should all be int):")
print(df_train_processed[CATEGORICAL_FEATURES].dtypes)

NaN values remaining: 0

Scaled column stats (should be ~mean=0, std=1):
      AMT_INCOME_TOTAL  AMT_CREDIT  AGE_YEARS
mean              -0.0        -0.0        0.0
std                1.0         1.0        1.0

Categorical dtypes (should all be int):
OCCUPATION_TYPE        int64
ORGANIZATION_TYPE      int64
NAME_EDUCATION_TYPE    int64
NAME_FAMILY_STATUS     int64
NAME_INCOME_TYPE       int64
CODE_GENDER            int64
dtype: object


## 5. Transform Test Data

Test data uses `transform()` — the already-fitted scaler and encoders. No refitting.

In [6]:
df_test_processed = preprocessor.transform(df_test, has_target=False)

print(f"Processed test shape: {df_test_processed.shape}")
print(f"NaNs in test: {df_test_processed.isnull().sum().sum()}")
df_test_processed.head()

Processed test shape: (48744, 20)
NaNs in test: 0


,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,LOAN_TO_GOODS_RATIO,EXT_SOURCE_1_MISSING,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING,AGE_YEARS,YEARS_EMPLOYED,YEARS_ID_PUBLISH,IS_UNEMPLOYED,OCCUPATION_TYPE,ORGANIZATION_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_INCOME_TYPE,CODE_GENDER
0,-0.142533,-0.075097,-0.451790,1.781131,1.441565,-2.038369,1.137631,0,0,0,0.734193,0.036230,-1.445696,0,17,28,1,1,7,0
1,-0.294354,-0.934825,-0.671924,0.435266,-1.167540,-0.473477,0.924803,0,0,0,0.464485,1.038013,-0.908413,0,9,42,4,1,7,1
2,0.142129,0.159601,2.943988,0.012103,0.970733,0.545375,-0.564994,1,0,0,0.916824,1.032864,0.337075,0,4,54,1,1,7,1
3,0.616567,2.424840,1.511720,0.153673,-0.025286,0.555178,-0.990650,0,0,0,-0.472274,-0.180511,0.804133,0,14,5,4,1,7,0
4,0.047242,0.065776,0.342121,-2.167493,-0.465323,0.112063,-0.990650,0,0,1,-0.686757,-0.028371,0.839908,0,17,5,4,1,7,1


## 6. Save Processed Data and Preprocessing Objects

In [7]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/mappings', exist_ok=True)

# Save processed data as parquet (faster + no index column problem)
df_train_processed.to_parquet('../data/processed/train_processed.parquet', index=False)
df_test_processed.to_parquet('../data/processed/test_processed.parquet', index=False)
print("Saved processed parquets.")

# Save scaler, encoders, and mappings
preprocessor.save(
    scaler_path='../models/scaler.pkl',
    encoders_path='../models/encoder_objects.pkl',
    mappings_dir='../data/mappings'
)

print("\nAll preprocessing objects saved.")

Saved processed parquets.
[preprocessor] Saved scaler → ../models/scaler.pkl
[preprocessor] Saved encoders → ../models/encoder_objects.pkl
[preprocessor] Saved mappings → ../data/mappings/

All preprocessing objects saved.


## 7. Inspect Mappings

Verify that the integer → label mappings look correct. These are what the frontend will use to display human-readable labels.

In [8]:
import json

with open('../data/mappings/OCCUPATION_TYPE.json') as f:
    occ_mapping = json.load(f)

print("OCCUPATION_TYPE mapping (int → label):")
for k, v in occ_mapping.items():
    print(f"  {k}: {v}")

OCCUPATION_TYPE mapping (int → label):
  0: Accountants
  1: Cleaning staff
  2: Cooking staff
  3: Core staff
  4: Drivers
  5: HR staff
  6: High skill tech staff
  7: IT staff
  8: Laborers
  9: Low-skill Laborers
  10: Managers
  11: Medicine staff
  12: Private service staff
  13: Realty agents
  14: Sales staff
  15: Secretaries
  16: Security staff
  17: Unknown
  18: Waiters/barmen staff


## Summary

| Object | Path | Purpose |
|--------|------|--------|
| Processed train | `data/processed/train_processed.parquet` | Input for training notebook |
| Processed test | `data/processed/test_processed.parquet` | Held-out evaluation |
| Scaler | `models/scaler.pkl` | Transform new data in API |
| Encoders | `models/encoder_objects.pkl` | Encode categories in API |
| Mappings | `data/mappings/*.json` | Decode integers in frontend |

**Next:** Run `03_model_training.ipynb` (on Google Colab with GPU).